# RG3 F1: 역사 가용성에 따른 전문가 학습/라우팅

19개가스/601개정상/7개관측구간 전체평가 유지. 역사3변수는 학습시 유효한 샷만 사용(정상/가스 모두 같은가용성 기준). 역사부족 평가샷은 현재값6 전문가로 예측. full10 또는min5가용성 기준·역사만/역사와현재평균·기존중앙값모델을 내부F1 선택. 전문가마다RF/ET2시드 평균. 절대날짜/RunID/평가라벨로 라우팅하지 않는다.

제한: 재사용된조건부순위캐시의정상 기준은 기존imputation 포함; 이번 대조는분류기학습/라우팅 단계만 바꾼다. 전체순위참조를완전기록만으로재학습한실험과 구분해야 한다. CN7미사용. 외부구간은 모델·라우팅선택·임계값 학습에서 제외. 새 독립 검증 아닌 개발용 중첩검증.


In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,precision_recall_curve,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
z=np.load(t/'rg3_F1_boost_fold_cache.npz',allow_pickle=False)
y=z['labels'];groups=z['groups'];times=z['timestamps'];blocks=np.unique(groups)
avail=pd.read_csv(t/'rg3_history_availability_all620.csv').set_index('TimeStamp').loc[times]
full=pd.read_csv(t/'rg3_full10_history_availability.csv').set_index('TimeStamp').loc[times]
assert len(y)==620 and y.sum()==19 and np.array_equal(avail.actual_gas,y)
masks=[avail.valid_noise3.to_numpy(bool),full.full10_noise3.to_numpy(bool)]
sources={tuple(z[k].astype(int)):k.split('_')[1] for k in z.files if k.startswith('ref_')}
memo={};audit=[]
def fit_score(F,ref):
 if len(np.unique(y[ref]))<2:return np.zeros(len(y))
 vals=[]
 for cls in [RandomForestClassifier,ExtraTreesClassifier]:
  for seed in [42,7]:
   m=cls(n_estimators=80,max_depth=4,min_samples_leaf=4,class_weight='balanced',random_state=seed,n_jobs=2)
   vals.append(m.fit(F[ref],y[ref]).predict_proba(F)[:,1])
 return np.mean(vals,axis=0)
def predict(ref,q):
 assert set(groups[ref]).isdisjoint(groups[q])
 key=tuple(ref)
 if key not in memo:
  n=sources[key];noise=z['features_'+n+'_1'][:,[6,8,9]];state=z['features_'+n+'_2'][:,:6]
  legacy=fit_score(noise,ref);current=fit_score(state,ref)
  vals=[legacy,current]
  for mask in masks:
   usable=ref[mask[ref]]
   history=fit_score(noise,usable)
   vals.append(np.where(mask,history,current))
   vals.append(np.where(mask,(history+current)/2,current))
   audit.append(dict(train=len(ref),history_train=len(usable),history_train_gas=int(y[usable].sum()),history_unavailable_gas=int(y[ref[~mask[ref]]].sum()),train_blocks=','.join(map(str,np.unique(groups[ref]))),excluded_blocks=','.join(map(str,np.setdiff1d(blocks,groups[ref])))))
  memo[key]=np.column_stack(vals)
 return memo[key][q]
def stats(yy,p):
 tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
 return dict(TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0,recall=tp/(tp+fn) if tp+fn else 0)
def cutfit(yy,score):
 p,r,c=precision_recall_curve(yy,score);f=np.divide(2*p[:-1]*r[:-1],p[:-1]+r[:-1],out=np.zeros_like(c),where=p[:-1]+r[:-1]>0)
 cut=float(c[np.lexsort((c,r[:-1],f))[-1]]);return cut,stats(yy,score>=cut)
names=['legacy_imputed_noise3','current_state6','available_history_route','available_history_mean','full10_history_route','full10_history_mean']
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups==b);ref=np.flatnonzero(groups!=b);out=predict(ref,q);inner=np.full((len(ref),6),np.nan)
 for ib in np.unique(groups[ref]):
  mask=groups[ref]==ib;inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for col,name in enumerate(names):
  cut,result=cutfit(y[ref],inner[:,col]);candidates.append((result['F1'],result['recall'],-result['FP'],col,cut))
  for j,i in enumerate(q):records.append(dict(model=name,TimeStamp=times[i],block=int(b),actual=int(y[i]),prediction=int(out[j,col]>=cut),score=float(out[j,col]),threshold=cut))
 best=max(candidates,key=lambda a:a[:3])
 for j,i in enumerate(q):records.append(dict(model='inner_F1_selected_coverage_route',TimeStamp=times[i],block=int(b),actual=int(y[i]),prediction=int(out[j,best[3]]>=best[4]),score=float(out[j,best[3]]),threshold=best[4]))
 choices.append(dict(block=int(b),selected=names[best[3]],threshold=best[4],inner_F1=best[0]))
 (t/'rg3_coverage_route_progress.json').write_text(str(int(b)),encoding='utf-8')
p=pd.DataFrame(records);m=pd.DataFrame([dict(model=name,AP=average_precision_score(g.actual,g.score),**stats(g.actual,g.prediction)) for name,g in p.groupby('model')])
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(audit),'training_audit')]:obj.to_csv(t/f'rg3_coverage_route_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))


                           model       AP  TP  FP  FN  TN       F1   recall
          available_history_mean 0.096510   7  66  12 535 0.152174 0.368421
         available_history_route 0.116407   7  38  12 563 0.218750 0.368421
                  current_state6 0.024267   6 305  13 296 0.036364 0.315789
             full10_history_mean 0.096585   7  54  12 547 0.175000 0.368421
            full10_history_route 0.124400   7  34  12 567 0.233333 0.368421
inner_F1_selected_coverage_route 0.136051   7  42  12 559 0.205882 0.368421
           legacy_imputed_noise3 0.125855   7  39  12 562 0.215385 0.368421
